# Model Debugging & Error Analysis — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*

Learn the workflow FAANG expects after you have a model: diagnose errors, find slices that fail, run feature ablations, and iterate safely.

**Outcome.** Students can:
- build a baseline churn classifier,
- compute slice metrics and identify failure cohorts,
- perform error analysis (FP vs FN),
- run feature ablation and interpret results,
- tune threshold for business cost.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Model Debugging & Error Analysis — Student Lab (Customer Churn)

You will build a baseline churn model, then debug it via slice analysis and feature ablation.

## Section 0 — Load churn dataset (with fallback)

Expected path: `data/churn/churn.csv`

The target column is commonly `Churn` (Yes/No) or `churn` (0/1).

In [ ]:
$10

**Why this works.** # Model Debugging & Error Analysis — Instructor (Customer Churn)

## Task 2: Build `y` robustly
*Section: Target normalization + holdout split*

## Section 1 — Target normalization + holdout split

### Task 1.1: Build `y` robustly

- Support both Yes/No and 0/1 churn targets
- Create X/y
- Train/holdout split (stratify)

In [ ]:
target_col = 'Churn' if 'Churn' in df.columns else ('churn' if 'churn' in df.columns else None)
if target_col is None:
    raise ValueError('Expected churn target column: Churn or churn')
y_raw = df[target_col]
if y_raw.dtype == object:
    y = y_raw.astype(str).str.lower().map({'yes': 1, 'no': 0})
else:
    y = y_raw.astype(int)
if y.isna().any():
    raise ValueError('Target contains unexpected values; expected Yes/No or 0/1.')
X = df.drop(columns=[target_col])

Xtr, Xho, ytr, yho = train_test_split(X, y.values, test_size=0.25, random_state=0, stratify=y.values)
print('train/holdout', Xtr.shape, Xho.shape)

## Task 3: Build ColumnTransformer + LogisticRegression
*Section: Baseline pipeline (leakage-safe)*

## Section 2 — Baseline pipeline (leakage-safe)

### Task 2.1: Build ColumnTransformer + LogisticRegression

- Identify numeric vs categorical columns
- Build preprocess pipeline
- Fit on train
- Report holdout accuracy + ROC-AUC

In [ ]:
num_cols = [c for c in Xtr.columns if pd.api.types.is_numeric_dtype(Xtr[c])]
cat_cols = [c for c in Xtr.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ]
)

pipe = Pipeline(steps=[('preprocess', preprocess), ('model', LogisticRegression(max_iter=5000))])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xho)
proba = pipe.predict_proba(Xho)[:, 1]
print('holdout_acc', accuracy_score(yho, pred))
print('holdout_auc', roc_auc_score(yho, proba))

# Rationale: use pipeline so preprocessing is fit only on training data; avoids leakage and makes debugging trustworthy.

**Why this works.** use pipeline so preprocessing is fit only on training data; avoids leakage and makes debugging trustworthy.

## Task 4: Confusion matrix and error types
*Section: Error breakdown (FP vs FN)*

## Section 3 — Error breakdown (FP vs FN)

### Task 3.1: Confusion matrix and error types

- Compute confusion matrix
- Count FP/FN
- Identify 5 examples of FP and FN rows from holdout

In [ ]:
cm = confusion_matrix(yho, pred)
print('confusion_matrix\n', cm)

ho = Xho.copy()
ho['y_true'] = yho
ho['p'] = proba
ho['y_pred'] = pred

fp = ho[(ho['y_true'] == 0) & (ho['y_pred'] == 1)]
fn = ho[(ho['y_true'] == 1) & (ho['y_pred'] == 0)]
print('fp', len(fp), 'fn', len(fn))

# Rationale: separate FP vs FN; fixes differ (thresholding, features, cohort-specific modeling).

**Why this works.** separate FP vs FN; fixes differ (thresholding, features, cohort-specific modeling).

### Figure: ROC Curve (Baseline)

![ROC Curve](/images/w5-d4-roc-curve.png)

### Figure: Confusion Matrix

![Confusion Matrix](/images/w5-d4-confusion-matrix.png)

### Figure: Predicted Probability by Class

![Probability Histogram](/images/w5-d4-proba-hist.png)

## Task 5: Slice metrics by key categorical columns
*Section: Slice analysis*

## Section 4 — Slice analysis

### Task 4.1: Slice metrics by key categorical columns

- For each of these (if present): Contract, InternetService, PaymentMethod
- Report: count, churn_rate, accuracy

**Checkpoint:** Why can slice failures happen even if overall metrics are good?

In [ ]:
def slice_table(df_holdout, col):
    def _slice_metrics(d):
        y_true = d['y_true'].values
        y_pred = d['y_pred'].values
        y_prob = d['p'].values
        auc = float('nan')
        if len(np.unique(y_true)) >= 2:
            auc = float(roc_auc_score(y_true, y_prob))
        return pd.Series({
            'n': len(d),
            'churn_rate': float(np.mean(y_true)),
            'acc': float(np.mean(y_true == y_pred)),
            'roc_auc': auc,
        })

    g = df_holdout.groupby(col).apply(_slice_metrics)
    return g.sort_values('n', ascending=False)

for col in ['Contract','InternetService','PaymentMethod']:
    if col in ho.columns:
        print('---', col)
        print(slice_table(ho, col))

# Rationale: slice analysis reveals systematic errors in cohorts; overall metrics can mask them.

**Why this works.** slice analysis reveals systematic errors in cohorts; overall metrics can mask them.

### Figure: Slice Performance

![Slice Performance](/images/w5-d4-slice-performance.png)

## Task 6: Drop feature groups and re-evaluate
*Section: Feature ablation*

## Section 5 — Feature ablation

### Task 5.1: Drop feature groups and re-evaluate

- Define groups by prefix or by a manual list
- Fit the same pipeline after dropping each group
- Report the delta in AUC

In [ ]:
$12

**Why this works.** ablation estimates marginal utility and helps you prioritize feature work.

### Figure: Feature Ablation Impact

![Ablation Delta](/images/w5-d4-ablation-delta.png)

## Task 7: Pick threshold for cost ratio
*Section: Thresholding with costs*

## Section 6 — Thresholding with costs

### Task 6.1: Pick threshold for cost ratio

Let cost(FN)=5 and cost(FP)=1.

- Sweep thresholds
- Compute cost on holdout
- Report best threshold

In [ ]:
# Thresholding with costs
cost_fp = 1.0
cost_fn = 5.0
ths = np.linspace(0.05, 0.95, 91)
best = None
for t in ths:
    yhat = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yho, yhat).ravel()
    cost = cost_fp * fp + cost_fn * fn
    if best is None or cost < best[0]:
        best = (float(cost), float(t), int(fp), int(fn))
print('best (cost, thr, fp, fn):', best)

# Rationale: threshold should align with business costs; not necessarily 0.5.

**Why this works.** threshold should align with business costs; not necessarily 0.5.

### Figure: Threshold vs Business Cost

![Threshold Cost](/images/w5-d4-threshold-cost-v2.png)